# Code Interpreter

*Notebook 12*

Let your agent write and run Python code in a secure sandboxed environment.

<br>
<br>

**Topics:**
- What Code Interpreter does and when to use it
- Enabling Code Interpreter with `CodeInterpreterTool`
- Data analysis workflows
- File input — uploading data to the container
- Math and reasoning with code execution

---

## 🔧 Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from openai import OpenAI
from agents import Agent, CodeInterpreterTool, Runner

MODEL = "gpt-5-mini"

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


print("✅ Ready!")

---

## 🎯 The Problem

Language models are good at reasoning about numbers but unreliable at actually computing them. Code Interpreter solves this by letting the agent write Python code and run it — so the answer comes from execution, not guesswork.

---

## 🖥️ Part 1: How Code Interpreter Works

When Code Interpreter is enabled:
1. The agent receives a task requiring computation
2. It writes Python code to solve it
3. OpenAI runs the code in an isolated sandbox container
4. The output is returned to the agent, which uses it to form its final response

The sandbox has no internet access and is discarded after the session — it's safe, isolated execution.

### 💡 Cost Note

Code Interpreter runs in a hosted sandbox container with a small per-session charge separate from token costs. This demo should cost very little — check current rates at [platform.openai.com/docs/pricing](https://platform.openai.com/docs/pricing).

---

## 🔢 Part 2: Math and Computation

The clearest use case for Code Interpreter is precise computation — the agent writes and runs the math instead of estimating.

### Before: Without Code Interpreter

In [ ]:
agent_no_code = Agent(
    name="NoCodeAgent",
    instructions="You are a helpful assistant.",
    model=MODEL
)

result = await Runner.run(agent_no_code, input="What is the sum of all prime numbers below 1000?")

print("Without Code Interpreter:")
print(result.final_output)

### After: With Code Interpreter

In [ ]:
agent_with_code = Agent(
    name="CodeAgent",
    instructions="You are a helpful assistant. Always use code to answer math and data questions.",
    model=MODEL,
    tools=[CodeInterpreterTool(tool_config={
        "type": "code_interpreter",
        "container": {"type": "auto"}
    })]
)

result = await Runner.run(agent_with_code, input="What is the sum of all prime numbers below 1000?")

print("With Code Interpreter:")
print(result.final_output)

### 💡 Why This Works

The agent writes Python to compute the answer, runs it in the sandbox, and reports the verified result — not an estimate.

Unlike `FileSearchTool`, `CodeInterpreterTool` takes a `tool_config` dict — `"container": {"type": "auto"}` tells the SDK to spin up a temporary Python sandbox automatically.

---

## 📊 Part 3: Data Analysis

Code Interpreter can analyze inline data — no file upload needed. Pass the data in the message and let the agent write the analysis code.

Use inline data for small examples or generated data; switch to file upload when the dataset is too large or awkward to paste into the prompt.

In [ ]:
sales_data = """
Month,Revenue,Units
January,42500,850
February,38200,764
March,51300,1026
April,49800,996
May,55100,1102
June,61200,1224
"""

analysis_agent = Agent(
    name="DataAnalyst",
    instructions="You are a data analyst. Use code to analyze data accurately and present clear summaries.",
    model=MODEL,
    tools=[CodeInterpreterTool(tool_config={
        "type": "code_interpreter",
        "container": {"type": "auto"}
    })]
)

# --------------------------------------------------------------
print("✅ Data analyst agent ready")

#### Run Data Analysis

In [ ]:
message = f"""Analyze this sales data and tell me:
1. Total revenue and units for the period
2. Best and worst performing months
3. Average monthly revenue
4. Month-over-month revenue growth trend

Data:
{sales_data}"""

result = await Runner.run(analysis_agent, input=message)

print("📊 DATA ANALYSIS RESULTS")
print("=" * 60)
print(result.final_output)
print("=" * 60)

#### Generate a Plot

The chart renders inside the sandbox — you'll see the agent's text description of what it generated, not an image. The agent can also save the plot as an image file (e.g., `plot.png`) inside the sandbox — useful for automated reporting where charts get exported, not displayed inline.

In [ ]:
plot_request = (
    "Using the sales data below, write Python code to create a bar chart "
    "showing monthly revenue. Label each bar with the month name. "
    "Describe what the chart shows.\n\n"
    f"Data:\n{sales_data}"
)

result = await Runner.run(analysis_agent, input=plot_request)

print("📊 PLOT GENERATION")
print("=" * 60)
print(result.final_output)
print("=" * 60)

### 💡 Why This Works

Code Interpreter can generate matplotlib plots in the sandbox — the agent writes the plotting code, runs it, and describes the output.

---

## 📁 Part 4: File Input

The container is a separate hosted environment — files on your local filesystem aren't automatically available inside it.

To give the agent access to a file, upload it via the Files API to get a file ID, then pass that ID into the container config (the dictionary you pass into `CodeInterpreterTool(...)` to tell the sandbox which files to mount before execution).

`FileSearchTool` and `CodeInterpreterTool` use different file backends. In Lesson 11, we uploaded files to a vector store for semantic search. Here, we upload to OpenAI's general file store — `purpose="assistants"` is the upload tier required for Code Interpreter and is unrelated to the (separate) Assistants API.

In [ ]:
# Create a sample CSV file locally
csv_path = Path("sales_report.csv")
csv_path.write_text(
    "Month,Revenue,Units\n"
    "January,42500,850\n"
    "February,38200,764\n"
    "March,51300,1026\n"
    "April,49800,996\n"
    "May,55100,1102\n"
    "June,61200,1224\n"
)

# Upload the file to OpenAI to get a file ID
with open(csv_path, "rb") as f:
    uploaded_file = openai_client.files.create(
        file=f,
        purpose="assistants"  # Required for Code Interpreter files — unrelated to the Assistants API
    )

file_agent = Agent(
    name="FileAnalyst",
    instructions="You are a data analyst. Read the uploaded CSV file and analyze it accurately using code.",
    model=MODEL,
    tools=[CodeInterpreterTool(tool_config={
        "type": "code_interpreter",
        "container": {
            "type": "auto",
            "file_ids": [uploaded_file.id]
        }
    })]
)

# --------------------------------------------------------------
print(f"✅ File uploaded: {uploaded_file.id}")
print(f"✅ File agent ready")

# Clean up local file
csv_path.unlink()
print("✅ Local copy removed — the file now lives in the container via uploaded_file.id")

#### Analyze the File

In [ ]:
message = (
    "Please read the uploaded CSV file (sales_report.csv) "
    "and tell me the total revenue, best month, and average monthly units."
)

result = await Runner.run(file_agent, input=message)

print("📁 FILE ANALYSIS RESULT")
print("=" * 60)
print(result.final_output)
print("=" * 60)

### 💡 Why This Works

Uploading the file via the Files API gives it a file ID. Passing that ID in `file_ids` makes the file available in the container's filesystem before the agent runs. The agent's code can then read it directly with pandas or the standard library.

---

## 💪 Practice Exercises

### Exercise 1: Statistics Calculator

*Covers: `CodeInterpreterTool` — numerical analysis in Python*

Create an agent that analyzes a list of numbers and reports key statistics.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Statistics Calculator
# --------------------------------------------------------------
# Objective: Use Code Interpreter to compute accurate statistics.

numbers = [23, 45, 12, 67, 34, 89, 56, 78, 90, 11, 44, 66, 33, 55, 77]

# TODO 1: Create an Agent with CodeInterpreterTool
#            Instructions: compute statistics precisely using code

# TODO 2: Run the agent asking for: mean, median, mode, std deviation,
#           min, max, and range of the numbers list

# TODO 3: Print result.final_output

# --- Write your code below this line ---

### Exercise 2: Text Analyzer

*Covers: `CodeInterpreterTool` — text analysis with Python*

Create an agent that analyzes a block of text and produces word frequency statistics.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Text Analyzer
# --------------------------------------------------------------
# Objective: Use code to count and rank word frequencies in text.

sample_text = """
Artificial intelligence is transforming how we work and live. Machine learning,
a subset of artificial intelligence, enables computers to learn from data.
Deep learning, a subset of machine learning, uses neural networks to process
complex patterns. Artificial intelligence and machine learning are now used
in healthcare, finance, and transportation.
"""

# TODO 1: Create an Agent with CodeInterpreterTool

# TODO 2: Run the agent asking it to:
#           - Count total words
#           - Find the top 5 most frequent words (excluding common words like 'a', 'the', 'of')
#           - Count unique words

# TODO 3: Print result.final_output

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

<br>
<br>
**Execution over estimation:**
- Code Interpreter runs Python in a secure sandbox — answers come from execution, not guesswork

- Ideal for math, statistics, data analysis, and any task where precision matters

<br>
<br>
**Inline data works:**
- Pass CSV or structured data directly in the message — no file upload needed for small datasets

- For large files, upload them directly to the container — shown in Part 4

---

## 📍 Next Step

**Notebook 13: Capstone #1**  

Combine web search, file search, and Code Interpreter into one agent that researches a topic, analyzes documents, and produces a structured report.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-12-code-interpreter)

---

##### Cleanup: Delete uploaded file

Run this cell after the exercises to remove the uploaded file from OpenAI.

In [ ]:
openai_client.files.delete(uploaded_file.id)
print(f"✅ Uploaded file deleted: {uploaded_file.id}")

---